In [87]:
# Notebook Setup
# Run this cell: 
# The lines below will instruct jupyter to reload imported modules before 
# executing code cells. This enables you to quickly iterate and test revisions
# to your code without having to restart the kernel and reload all of your 
# modules each time you make a code change in a separate python file.

%load_ext autoreload
%autoreload 2

import os

# Change path to project root
if os.getcwd().endswith("notebooks"):
    os.chdir(os.path.dirname(os.getcwd()))
print(os.getcwd())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/Users/shloknatarajan/stanford/research/daneshjou/AutoGKB


In [88]:
import json

In [89]:
# load persistent_data/llm_outputs/combined_output_11_02_25.json

with open("persistent_data/llm_outputs/combined_output_11_02_25.json", "r") as f:
    combined_output = json.load(f)

print(f"Loaded {len(combined_output)} records from combined_output_11_02_25.json")


Loaded 32 records from combined_output_11_02_25.json


In [90]:
combined_output['PMC5508045'].keys()

dict_keys(['var_pheno_ann', 'var_drug_ann', 'var_fa_ann', 'study_parameters', 'title', 'pmid', 'pmcid', 'summary', 'input_text', 'timestamp', 'prompts_used'])

In [91]:
# drop input_text, timestamp, and prompts_used
for pmcid in combined_output:
    for key in ['input_text', 'timestamp', 'prompts_used']:
        combined_output[pmcid].pop(key)


In [92]:
# add pmcid
for pmcid in combined_output:
    combined_output[pmcid]['pmcid'] = pmcid


In [99]:
import ftfy
import unicodedata
import re

def clean_string(s):
    """
    Remove problematic quote/backslash characters and normalize unicode to closest ASCII (if possible).
    """
    if not isinstance(s, str):
        return s
    
    # First, handle literal backslash-quote sequences
    # Use regex to replace backslash followed by quote (handles all cases)
    clean_s = re.sub(r'\\"', '"', s)  # Replace \" with "
    clean_s = re.sub(r"\\'", "'", clean_s)  # Replace \' with '
    
    # Strip leading and trailing quotes (common in citations that start/end with quotes)
    clean_s = clean_s.strip('"').strip("'")
    
    # Fix unicode issues (handles things like smart quotes, en dashes, etc.)
    clean_s = ftfy.fix_text(clean_s)
    
    # Convert common unicode characters to ASCII equivalents
    # En dash and em dash to regular dash
    clean_s = clean_s.replace('\u2013', '-').replace('\u2014', '-')
    # Multiplication sign to x
    clean_s = clean_s.replace('\u00d7', 'x')
    # Various quote types to regular quotes
    clean_s = clean_s.replace('\u2018', "'").replace('\u2019', "'")  # Left/right single quotes
    clean_s = clean_s.replace('\u201c', '"').replace('\u201d', '"')  # Left/right double quotes
    # Non-breaking space to regular space
    clean_s = clean_s.replace('\u00a0', ' ')
    
    # Normalize unicode characters (decompose combined characters)
    clean_s = unicodedata.normalize('NFKD', clean_s)
    
    # Remove any remaining backslashes
    clean_s = clean_s.replace("\\", "")
    
    # Remove control characters except newlines, carriage returns, and tabs
    clean_s = ''.join(char for char in clean_s if unicodedata.category(char)[0] != 'C' or char in '\n\r\t')
    
    # Clean up any weird whitespace
    clean_s = re.sub(r'[\u200b-\u200d\ufeff]', '', clean_s)  # Remove zero-width spaces
    
    return clean_s


In [103]:
def clean_all_strings(obj):
    """Recursively clean all string values in a nested dictionary/list structure."""
    if isinstance(obj, dict):
        return {key: clean_all_strings(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [clean_all_strings(item) for item in obj]
    elif isinstance(obj, str):
        return clean_string(obj)
    else:
        return obj

# Clean all string fields in all annotation types
for pmcid in combined_output:
    combined_output[pmcid] = clean_all_strings(combined_output[pmcid])

In [110]:
key_order = ['title', 'pmcid', 'summary', 'study_parameters', 'var_drug_ann', 'var_pheno_ann', 'var_fa_ann']
# order the keys in the dictionary
for pmcid in combined_output:
    combined_output[pmcid] = {k: combined_output[pmcid][k] for k in key_order}


In [111]:
# save each record as a separate json file
for pmcid in combined_output:
    base_path = "persistent_data/llm_outputs/split"
    if not os.path.exists(base_path):
        os.makedirs(base_path)
    with open(f"{base_path}/{pmcid}.json", "w", encoding='utf-8') as f:
        json.dump(combined_output[pmcid], f, indent=4, ensure_ascii=False)

In [112]:
# Do the same for the benchmark annotations
with open("persistent_data/benchmark_annotations.json", "r") as f:
    benchmark_annotations = json.load(f)

# add pmcid
for pmcid in benchmark_annotations:
    benchmark_annotations[pmcid]['pmcid'] = pmcid

key_order = ['title', 'pmid', 'pmcid', 'study_parameters', 'var_drug_ann', 'var_pheno_ann', 'var_fa_ann']

for pmcid in benchmark_annotations:
    benchmark_annotations[pmcid] = {k: benchmark_annotations[pmcid][k] for k in key_order}

cleaned_benchmark_annotations = clean_all_strings(benchmark_annotations)

# save each record as a separate json file
for pmcid in benchmark_annotations:
    base_path = "persistent_data/benchmark_annotations"
    if not os.path.exists(base_path):
        os.makedirs(base_path)
    with open(f"{base_path}/{pmcid}.json", "w", encoding='utf-8') as f:
        json.dump(cleaned_benchmark_annotations[pmcid], f, indent=4, ensure_ascii=False)


In [115]:
# Create markdown files for each record
import shutil
for pmcid in benchmark_annotations:
    # Get the markdown file from data/articles
    markdown_path = f"data/articles/{pmcid}.md"
    # copy the markdown file to the benchmark_annotations directory
    os.makedirs(f"persistent_data/benchmark_markdown", exist_ok=True)
    if os.path.exists(markdown_path):
        shutil.copy(markdown_path, f"persistent_data/benchmark_markdown/{pmcid}.md")
    else:
        print(f"Markdown file for {pmcid} does not exist")


Markdown file for PMC12260932 does not exist
